# EFSM Full Speech Demo on Kaggle

This notebook launches the final Gradio demo for the EFSM project.

Real demo path:

`microphone audio -> Whisper ASR -> Qwen2.5-7B-Instruct + EFSM LoRA checkpoint-2667 -> TTS audio reply`

Use `--mock` only to test the UI. Remove `--mock` for the actual fine-tuned 7B model demo.

## 0. Kaggle Settings

Before running the cells:

- Turn **Internet** on.
- Select a GPU accelerator, preferably **T4 x2**.
- Add a Kaggle secret named `HF_TOKEN` if the Hugging Face checkpoint repo is private.

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

try:
    secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets.")
except Exception as exc:
    print("HF_TOKEN was not loaded. This is okay only if all Hugging Face repos are public.")
    print(type(exc).__name__, exc)

## 1. Clone the Project Repo

In [ ]:
!rm -rf empathetic-voice-llm
!git clone https://github.com/tasbidrahman10/empathetic-voice-llm.git
%cd empathetic-voice-llm
!git log --oneline -1

## 2. Install System Audio Dependencies

`pyttsx3` uses the Linux speech engine under the hood, so Kaggle needs `espeak`. `ffmpeg` is useful for audio conversion.

In [ ]:
!apt-get update -qq
!apt-get install -y espeak ffmpeg

## 3. Install Python Dependencies

Kaggle usually already has CUDA-enabled PyTorch. The project requirements intentionally do not reinstall `torch`.

In [ ]:
!pip install -q -r requirements.txt

## 4. Quick GPU Check

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU 0:", torch.cuda.get_device_name(0))

## 5. Optional UI Mock Test

Run this first if you want to confirm that Gradio opens correctly. This does **not** load the fine-tuned model.

Stop this cell before running the real demo.

In [ ]:
!python demo/app.py --mock --share

## 6. Real Full Demo

This is the actual project demo. It loads:

- `Qwen/Qwen2.5-7B-Instruct`
- LoRA adapter repo `tasbid001/efsm-checkpoints-fixed`
- adapter subfolder `checkpoint-2667`
- `openai/whisper-base` for ASR

When Gradio prints `Running on public URL: https://...gradio.live`, open that link to demonstrate the project.

In [ ]:
!python demo/app.py --share

## 7. If Kaggle Runs Out of Memory

Try a shorter response first:

In [ ]:
!python demo/app.py --share --max-new-tokens 100

Or test an earlier checkpoint:

In [ ]:
!python demo/app.py --share --adapter-subfolder checkpoint-1800 --max-new-tokens 100